# WEEK 3：面向对象编程（OOP）学习与练习

依据 [OOP approach Ver2.docx](source/OOP%20approach%20Ver2.docx) 整理。本笔记把原文混合的 C / Java 风格伪代码统一改写为 Python，并重点补全新版的 **Employee、PaymentProcessor 和 ABC 抽象类**。

学习目标：理解函数如何减少重复、返回值如何减少共享状态；掌握类、对象、封装、继承、多态与抽象；能用 `property` 维护数据约束，解释下划线和抽象类的真实行为。

**运行方式：**选择 Python 3 内核，依次运行；复习时使用 **Restart Kernel and Run All Cells**。仅使用 Python 标准库，不需要安装第三方库、联网或输入。所有预期异常均已捕获；练习后紧跟参考实现，可以先折叠答案再尝试。

## 1. 先看整条设计路线

重复语句 → 提取函数 → 参数与局部变量 → `return` → 通用参数 → 类管理状态 → 继承与多态 → 抽象约定。

| 原文的表达 | 本笔记的 Python 写法 |
|---|---|
| `this`、`new`、构造函数 | `self`、`ClassName(...)`、`__init__` 初始化方法 |
| `private` / `protected` | `_name` 约定、`__name` 名称改写；没有同等的访问修饰符 |
| `extend`、`result(...)` | `class Child(Parent):`、`return ...` |
| `Sum_Power` / `Sum_Powers` | 统一为 `SumPower`，避免原文命名不一致 |

**设计提醒：**原文把 OOP 描述得很万能，但函数、模块和清晰的接口也能让程序易维护。简单计算通常一个通用函数就足够；需要长期保存配置或状态、组织相关行为时，再考虑类。

## 2. 重复代码：变化时容易漏改

两组数据都要计算平方和。如果把同样的表达式复制到很多地方，将来改成三次方和时就得逐处修改。先预测下面输出，再运行。

In [1]:
a1, b1 = 2, 3
result_1 = a1 * a1 + b1 * b1

a2, b2 = 4, 5
result_2 = a2 * a2 + b2 * b2

print("第一组平方和：", result_1)
print("第二组平方和：", result_2)

第一组平方和： 13
第二组平方和： 41


输出是 `13` 和 `41`。把公共计算放进函数，可以集中维护公式；但**函数如果把结果写到全局变量，仍会留下共享状态问题**。

### 2.1 全局变量：后一调用覆盖共享位置

这里的 `global shared_result` 明确表示修改函数外的变量。第二次调用后，直接读取 `shared_result` 只能得到最新结果。问题在于结果共用同一位置，不是 Python 会把所有以前保存的整数都改掉。

In [2]:
shared_result = None

def sum_squares_global(a, b):
    global shared_result
    left = a ** 2  # left、right 只属于这次函数调用
    right = b ** 2
    shared_result = left + right

sum_squares_global(2, 3)
saved_first = shared_result
print("第一次调用后，共享位置：", shared_result)

sum_squares_global(4, 5)
print("第二次调用后，共享位置：", shared_result)
print("另外保存的第一次整数结果：", saved_first)

第一次调用后，共享位置： 13
第二次调用后，共享位置： 41
另外保存的第一次整数结果： 13


### 2.2 用参数传入数据，用 `return` 交回结果

全局版本产生 `13 → 41` 的覆盖；单独保存的整数 `saved_first` 仍为 `13`。更清楚的接口是：输入由参数提供，输出通过 `return` 提供，调用者自行命名和保存。

`print` 只是显示信息；`return` 才让调用者得到可继续运算的值。没有显式 `return` 的函数默认返回 `None`。

In [3]:
def sum_squares(a, b):
    left = a ** 2
    right = b ** 2
    return left + right

first = sum_squares(2, 3)
second = sum_squares(4, 5)
print("分别保存：", first, second)
print("两个结果继续相加：", first + second)

分别保存： 13 41
两个结果继续相加： 54


结果仍是 `13` 和 `41`，相加得到 `54`。现在函数不修改共享结果，调用关系更容易推理。

### 2.3 幂次变化：先把变化变成参数

不必为平方、立方、四次方各写一个几乎相同的函数。把 `power_order` 作为参数，即可复用 `a ** power_order + b ** power_order`。

下面为了让规则容易理解，只接受**非负整数幂次**；这是一项教学设计限制，不是 Python 幂运算本身的限制。特别排除 `bool`，因为在 Python 中 `bool` 是 `int` 的子类。

In [4]:
def validate_power_order(power_order):
    if isinstance(power_order, bool) or not isinstance(power_order, int):
        raise TypeError("幂次必须是整数，且不能是 bool")
    if power_order < 0:
        raise ValueError("本示例只接受非负整数幂次")
    return power_order

def sum_power(a, b, power_order=2):
    power_order = validate_power_order(power_order)
    return a ** power_order + b ** power_order

for n in range(2, 6):
    print(f"2^{n} + 3^{n} = {sum_power(2, 3, n)}")

2^2 + 3^2 = 13
2^3 + 3^3 = 35
2^4 + 3^4 = 97
2^5 + 3^5 = 275


输出依次为 `13、35、97、275`。如果任务只是偶尔计算这些值，`sum_power` 已经够用。

## 3. 类与对象：把配置和行为放在一起

如果某个“计算器”会反复使用同一幂次，可以把该配置保存在对象中。

| 概念 | 对应示例 |
|---|---|
| 类（class） | `SumPower`，描述对象的结构与行为 |
| 对象 / 实例（instance） | `SumPower(2)` 创建的一台平方和计算器 |
| 实例属性 | `self._power_order`，每个实例各自保存 |
| 方法（method） | `calculate` 和 `_power` |
| `self` | 当前接收方法调用的对象，不是另一个类 |
| `__init__` | 新实例创建后的初始化方法；给对象设置初始状态 |

`calculator.calculate(2, 3)` 调用时，Python 会把 `calculator` 自动传给 `self`。单下划线表示内部实现约定。

In [5]:
class SumPower:
    def __init__(self, power_order):
        self._power_order = validate_power_order(power_order)

    def _power(self, value):
        return value ** self._power_order

    def calculate(self, a, b):
        return self._power(a) + self._power(b)

square_calculator = SumPower(2)
cube_calculator = SumPower(3)

print("平方和：", square_calculator.calculate(2, 3))
print("立方和：", cube_calculator.calculate(2, 3))
print("方法调用的等价形式：", SumPower.calculate(square_calculator, 2, 3))

平方和： 13
立方和： 35
方法调用的等价形式： 13


### 3.1 实例各自保存状态

上面两个对象使用同一个类的代码，却分别保存幂次 `2` 和 `3`，因此输出 `13` 与 `35`。等价形式也返回 `13`，帮助理解 `self`。

下面一次建立四个对象。同一 `calculate` 接口可以复用，而不需要每次都向该方法传入幂次。`is` 判断是否为同一个对象；`vars` 在本示例中显示实例属性，仅用来观察内部状态。

In [6]:
calculators = [SumPower(n) for n in range(2, 6)]
print("四台计算器的结果：", [calc.calculate(2, 3) for calc in calculators])
print("平方计算器与立方计算器是同一对象吗？", square_calculator is cube_calculator)
print("平方计算器的实例属性：", vars(square_calculator))
print("立方计算器的实例属性：", vars(cube_calculator))

四台计算器的结果： [13, 35, 97, 275]
平方计算器与立方计算器是同一对象吗？ False
平方计算器的实例属性： {'_power_order': 2}
立方计算器的实例属性： {'_power_order': 3}


结果为 `[13, 35, 97, 275]`，对象身份比较为 `False`，两个实例的 `_power_order` 各不相同。类没有消灭参数，只是把经常重复的配置保存到实例里。

## 4. Employee：公开成员、单下划线与双下划线

新版教材用 `Employee` 说明 Python 的命名习惯：

| 写法 | 含义 | 能否直接从外部访问 |
|---|---|---|
| `name` | 公开属性 | 可以 |
| `_age` | 约定为内部属性 | 可以，但调用者通常应尊重约定 |
| `__salary` | 触发名称改写（name mangling） | `emp.__salary` 通常找不到；仍可用改写后的名字访问 |

单下划线不等于强制的 `protected`。双前导下划线主要帮助避免子类与父类的属性意外重名；不要把 `__init__` 这样的双尾下划线特殊方法与它混淆。

In [7]:
class Employee:
    def __init__(self, name, age, salary):
        self.name = name
        self._age = age
        self.__salary = salary

    def get_salary(self):
        return self.__salary

employee = Employee("Alice", 30, 85000)
print("公开姓名：", employee.name)
print("内部年龄（演示可访问性）：", employee._age)
print("通过公开方法读取工资：", employee.get_salary())

公开姓名： Alice
内部年龄（演示可访问性）： 30
通过公开方法读取工资： 85000


### 4.1 名称改写不是安全边界

结果为 `Alice、30、85000`。类内部的 `__salary` 被改写为 `_Employee__salary`；这是名称处理，不是加密或权限控制。

下一格先捕获直接访问失败，再展示改写后的名字。**后者只用于解释机制，业务代码应使用公开接口。**如果有人主动绕开接口，Python 命名约定不会阻止他；敏感数据安全不能仅依赖双下划线。

In [8]:
try:
    print(employee.__salary)
except AttributeError as exc:
    print("预期异常：", type(exc).__name__)

print("对象实际保存的属性：", vars(employee))
print("改写后的名称仍可访问：", employee._Employee__salary)
print("正常用法：", employee.get_salary())

预期异常： AttributeError
对象实际保存的属性： {'name': 'Alice', '_age': 30, '_Employee__salary': 85000}
改写后的名称仍可访问： 85000
正常用法： 85000


## 5. 用 property 维护数据约束

工资应当是有限的非负数。单纯写 `self.salary = salary` 会允许调用者以后写入负数；在 setter 中集中校验，可以让初始化和正常赋值经过同一规则。

`@property` 定义读取行为，`@salary.setter` 定义赋值行为。使用者仍写 `employee.salary`，不用记住额外的 `get_salary()` / `set_salary()` 名称。这里用新的 `ValidatedEmployee` 类，避免覆盖前面的演示。

为保持示例简单，工资接受 `int` / `float` 并转为浮点数；真实货币计算通常要另外设计精度、舍入和币种规则。

In [9]:
from math import isfinite

class ValidatedEmployee:
    def __init__(self, name, salary):
        self.name = name
        self.salary = salary  # 调用 setter；初始化也校验

    @property
    def salary(self):
        return self._salary

    @salary.setter
    def salary(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("工资必须是数字，且不能是 bool")
        if not isfinite(value) or value < 0:
            raise ValueError("工资必须是有限的非负数")
        self._salary = float(value)  # 全部检查通过后再修改状态

    def give_raise(self, amount):
        if isinstance(amount, bool) or not isinstance(amount, (int, float)):
            raise TypeError("加薪额必须是数字，且不能是 bool")
        if not isfinite(amount) or amount < 0:
            raise ValueError("加薪额必须是有限的非负数")
        self.salary = self.salary + amount

validated_employee = ValidatedEmployee("Alice", 85000)
validated_employee.give_raise(5000)
print(f"{validated_employee.name} 加薪后：{validated_employee.salary:.2f}")

Alice 加薪后：90000.00


### 5.1 合法更新成功，非法更新保留旧值

工资由 `85000` 变为 `90000`。下面尝试负数、非有限数和字符串，都会捕获异常；由于先校验、后赋值，失败不会破坏已保存的工资。初始化也调用 setter，因此不能创建负工资的实例。

这就是封装的一个实际用途：把数据和相关操作放在一起，并通过公开接口维护约束。调用者仍能主动写 `_salary` 绕开 setter，所以这是合作式接口设计，不是安全隔离。

In [10]:
for invalid_salary in [-1, float("nan"), float("inf"), "很多钱"]:
    before = validated_employee.salary
    try:
        validated_employee.salary = invalid_salary
    except (TypeError, ValueError) as exc:
        print(f"拒绝 {invalid_salary!r}：{type(exc).__name__}；工资仍为 {validated_employee.salary:.2f}")
    assert validated_employee.salary == before

try:
    ValidatedEmployee("Bob", -100)
except ValueError as exc:
    print("初始化也拒绝无效工资：", exc)

拒绝 -1：ValueError；工资仍为 90000.00
拒绝 nan：ValueError；工资仍为 90000.00
拒绝 inf：ValueError；工资仍为 90000.00
拒绝 '很多钱'：TypeError；工资仍为 90000.00
初始化也拒绝无效工资： 工资必须是有限的非负数


## 6. 继承与 super：在已有行为上扩展

`SumPowerWithPrint` 也是一台 `SumPower` 计算器，所以可以继承计算功能，只补充标签和显示功能。

`super().__init__(power_order)` 按方法解析顺序调用下一处初始化实现；在本节的单继承结构中，就是调用父类初始化。子类定义自己的 `__init__` 时，不会自动帮你执行父类的 `__init__`，因此这里需要显式调用。

In [11]:
class SumPowerWithPrint(SumPower):
    def __init__(self, power_order, label):
        super().__init__(power_order)
        self.label = label

    def print_power_order(self):
        print(f"{self.label}：幂次为 {self._power_order}")

named_calculator = SumPowerWithPrint(3, "立方和计算器")
named_calculator.print_power_order()
print("继承的计算功能：", named_calculator.calculate(2, 3))
print("属于 SumPower 吗？", isinstance(named_calculator, SumPower))

立方和计算器：幂次为 3
继承的计算功能： 35
属于 SumPower 吗？ True


输出显示幂次 `3`、计算结果 `35`、`isinstance` 为 `True`。`calculate` 没有在子类中重复定义，直接复用父类实现。

## 7. 多态：相同调用，不同实现

新版教材的 `PaymentProcessor` 定义 `process_payment(amount)`，`CreditCard` 与 `PayPal` 各自实现。调用方只关心对象支持这个行为，不必逐个判断支付方式。

以下只是字符串演示，不连接支付服务，也不会产生真实交易。父类抛出 `NotImplementedError` 表示“这个操作需要子类实现”，但这种写法本身不会禁止创建父类实例。

In [12]:
class PaymentProcessor:
    def process_payment(self, amount):
        raise NotImplementedError("子类需要实现 process_payment")

class CreditCard(PaymentProcessor):
    def process_payment(self, amount):
        return f"信用卡模拟付款：${amount:.2f}"

class PayPal(PaymentProcessor):
    def process_payment(self, amount):
        return f"PayPal 模拟付款：${amount:.2f}"

def checkout(processor, amount):
    return processor.process_payment(amount)

payments = [CreditCard(), PayPal()]
for payment in payments:
    print(checkout(payment, 100))

信用卡模拟付款：$100.00
PayPal 模拟付款：$100.00


### 7.1 幂求和中的方法重写

上一格的两行输出金额相同、支付方式不同。多态也能对应原文的平方与立方子类：父类提供 `calculate` 的流程，子类重写 `_power` 这一变化点。

这里使用 `_power`，让父类调用可以找到子类的重写方法。如果父子类各自定义 `__power`，它们会被改写成不同名字，容易破坏这个设计。仍要记住：只有幂次不同的简单需求，用前面的参数化 `SumPower` 通常更直接。

In [13]:
class PowerTemplate:
    def _power(self, value):
        raise NotImplementedError("子类需要实现 _power")

    def calculate(self, a, b):
        return self._power(a) + self._power(b)

class SumSquares(PowerTemplate):
    def _power(self, value):
        return value * value

class SumCubes(PowerTemplate):
    def _power(self, value):
        return value * value * value

for calculator in [SumSquares(), SumCubes()]:
    print(f"{type(calculator).__name__}：{calculator.calculate(2, 3)}")

SumSquares：13
SumCubes：35


### 7.2 `NotImplementedError` 在什么时候出现？

平方与立方子类分别得到 `13` 和 `35`。但如果漏写方法，普通父类并不会在创建对象时阻止你。下面的 `MissingPaymentProcessor()` 能成功实例化；只有调用继承来的占位方法时才抛出 `NotImplementedError`。

这为下一节的 `ABC` 提供对照：二者失败的时机不同。

In [14]:
class MissingPaymentProcessor(PaymentProcessor):
    pass

missing_payment = MissingPaymentProcessor()
print("普通继承：对象已经成功创建。")
try:
    checkout(missing_payment, 100)
except NotImplementedError as exc:
    print("调用方法时才失败：", type(exc).__name__, str(exc))

普通继承：对象已经成功创建。
调用方法时才失败： NotImplementedError 子类需要实现 process_payment


## 8. ABC 与 abstractmethod：阻止不完整的类实例化

Python 用标准库 `abc` 表达抽象基类：继承 `ABC`，并给必须由具体子类提供的方法加上 `@abstractmethod`。

**准确的失败时机：**未实现所有抽象方法的子类可以被定义，也可以继续被继承；当尝试创建它的实例时，才会出现 `TypeError`。这修正了原文“未实现就立即运行时报错”的模糊表述。

抽象类可以同时提供具体方法。下面 `description()` 是可复用的实现，`drive()` 是需要补全的行为。ABC 管理的是抽象方法是否仍未实现，不会自动证明实现正确或严格检查方法签名兼容性。

In [15]:
from abc import ABC, abstractmethod

class Vehicles(ABC):
    def description(self):
        return "这是一种交通工具"

    @abstractmethod
    def drive(self):
        pass

class IncompleteVehicle(Vehicles):
    pass  # 定义类本身不报错

class Car(Vehicles):
    def drive(self):
        return "汽车沿道路行驶"

print("IncompleteVehicle 类已成功定义。")
for vehicle_class in [Vehicles, IncompleteVehicle]:
    try:
        vehicle_class()
    except TypeError as exc:
        print(f"实例化 {vehicle_class.__name__} 时失败：{type(exc).__name__}")

car = Car()
print(car.description())
print(car.drive())

IncompleteVehicle 类已成功定义。
实例化 Vehicles 时失败：TypeError
实例化 IncompleteVehicle 时失败：TypeError
这是一种交通工具
汽车沿道路行驶


## 9. 抽象、封装、继承、多态不要混在一起

上一格中两个不完整类均在**实例化时**失败；`Car` 实现 `drive` 后可以创建，还能复用 `description`。

| 概念 | 主要关注什么 | 本笔记中的具体例子 |
|---|---|---|
| 封装 Encapsulation | 数据与操作如何组织，哪些规则应经过公开接口维护 | `ValidatedEmployee.salary` 校验工资 |
| 抽象 Abstraction | 调用者需要知道什么，把哪些实现细节藏在接口之后 | 调用 `checkout` 时只需知道 `process_payment` |
| 继承 Inheritance | 如何建立父子关系并复用 / 扩展行为 | `SumPowerWithPrint(SumPower)` |
| 多态 Polymorphism | 同一个操作如何使用不同对象的实现 | 循环调用信用卡与 PayPal 的付款方法 |

抽象不一定需要抽象类：一个清晰的函数也能隐藏复杂细节。封装不只是“把变量设为私有”，关键是让状态及其操作有一致的规则。

**抽象类与接口：**原文的“是什么 / 能做什么”可以作为设计提示，但不是 Python 的语法分类。Python 没有 `interface` 关键字，常用鸭子类型、ABC，或类型标注体系中的 `typing.Protocol` 描述接口。原文“接口只能包含抽象方法”的说法依赖具体语言，不能作为普遍规则；本笔记不把 ABC 等同于只有空方法的接口。

## 10. 练习 1：通用函数还是类？

**题目：**给定数字列表 `[1, 2, 3]`，计算每个数字的三次方之和。先写函数 `sum_many_powers(values, power_order)`；再写保存幂次的 `PowerSeries` 类，其 `calculate(values)` 方法复用函数。

要求：沿用前面的幂次校验；空列表返回 `0`；结果应为 `36`。思考：只调用一次时选哪一种？反复使用固定幂次时，类带来了什么便利？

下一格为参考实现。可以先在它上方插入自己的代码格。

In [16]:
def sum_many_powers(values, power_order):
    power_order = validate_power_order(power_order)
    return sum(value ** power_order for value in values)

class PowerSeries:
    def __init__(self, power_order):
        self._power_order = validate_power_order(power_order)

    def calculate(self, values):
        return sum_many_powers(values, self._power_order)

series = PowerSeries(3)
print("函数结果：", sum_many_powers([1, 2, 3], 3))
print("对象结果：", series.calculate([1, 2, 3]))
print("空列表：", series.calculate([]))
assert series.calculate([1, 2, 3]) == 36
assert series.calculate([]) == 0

函数结果： 36
对象结果： 36
空列表： 0


两种写法都返回 `36`；空列表为 `0`。一次计算用函数最直接，对象的价值在于保存可复用配置，不在于“写成类就一定更好”。

## 11. 练习 2：把状态限制在合理范围

**题目：**创建 `StudentRecord` 类，保存姓名与 `score`。通过 `property` 限制分数为 `0` 到 `100` 的有限数字，排除布尔值。初始化也必须校验，非法赋值后保留原成绩。

预测：初始成绩 `80`，改为 `95`，再尝试 `101`、`-1` 和 `True`，最后成绩是多少？下一格为参考实现。

In [17]:
class StudentRecord:
    def __init__(self, name, score):
        self.name = name
        self.score = score

    @property
    def score(self):
        return self._score

    @score.setter
    def score(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("分数必须是数字，且不能是 bool")
        if not isfinite(value) or not 0 <= value <= 100:
            raise ValueError("分数必须在 0 到 100 之间")
        self._score = float(value)

student = StudentRecord("Mina", 80)
student.score = 95
for invalid_score in [101, -1, True]:
    try:
        student.score = invalid_score
    except (TypeError, ValueError) as exc:
        print(f"拒绝 {invalid_score!r}：{type(exc).__name__}")

print("最终成绩：", student.score)
assert student.score == 95

拒绝 101：ValueError
拒绝 -1：ValueError
拒绝 True：TypeError
最终成绩： 95.0


最终成绩为 `95.0`。验证通过后才赋值，因此三个失败更新都没有改变原有状态。

## 12. 练习 3：给支付接口增加实现

**题目：**把支付父类改成 ABC，要求所有具体子类实现 `process_payment(amount)`；创建 `BankTransfer` 和 `DigitalWallet` 两个子类。复用已有 `checkout` 函数处理它们，然后演示一个漏实现方法的子类在实例化时失败。

这里只练习接口和多态，不扩展金额校验或真实支付逻辑。下一格为参考实现。

In [18]:
class AbstractPaymentProcessor(ABC):
    @abstractmethod
    def process_payment(self, amount):
        pass

class BankTransfer(AbstractPaymentProcessor):
    def process_payment(self, amount):
        return f"银行转账模拟付款：${amount:.2f}"

class DigitalWallet(AbstractPaymentProcessor):
    def process_payment(self, amount):
        return f"电子钱包模拟付款：${amount:.2f}"

class UnfinishedPayment(AbstractPaymentProcessor):
    pass

for processor in [BankTransfer(), DigitalWallet()]:
    print(checkout(processor, 120))

try:
    UnfinishedPayment()
except TypeError as exc:
    print("缺少实现，实例化失败：", type(exc).__name__)

银行转账模拟付款：$120.00
电子钱包模拟付款：$120.00
缺少实现，实例化失败： TypeError


两种新方式都能通过原来的 `checkout` 处理金额 `120`，调用方无需修改。这也说明 Python 可以根据对象支持的行为使用它；`checkout` 并没有要求这些对象必须继承最初的 `PaymentProcessor`。

## 13. 复习速查与常见误区

| 常见误区 | 更准确的理解 |
|---|---|
| 全局变量被覆盖，就连以前保存的整数都变了 | 共享位置变了；本例另存的整数结果不变 |
| 所有问题都应该用 OOP | 先考虑函数；需要组织状态和相关行为时再选择类 |
| `_age` 只能在子类中访问 | 单下划线是内部使用约定，不强制阻止外部访问 |
| `__salary` 绝对无法读取 | 它被名称改写，不能充当安全边界 |
| setter 失败也会修改状态 | 是否修改取决于实现；本例先校验、后赋值 |
| `NotImplementedError` 会禁止实例化 | 普通类仍能实例化；调用占位方法时才失败 |
| 抽象方法没实现，定义子类马上失败 | ABC 允许定义不完整子类，禁止其实例化 |
| 抽象等于封装 | 抽象简化使用接口；封装组织数据、行为和约束 |

下一格做少量学习检查：验证函数 / 对象结果一致、独立实例保存各自配置、合法状态与多态输出符合预期。

In [19]:
assert sum_power(2, 3, 2) == SumPower(2).calculate(2, 3) == 13
assert SumPower(3).calculate(2, 3) == 35
assert square_calculator.calculate(2, 3) == 13
assert cube_calculator.calculate(2, 3) == 35
assert validated_employee.salary == 90000
assert isinstance(named_calculator, SumPower)
assert checkout(CreditCard(), 10) == "信用卡模拟付款：$10.00"
assert checkout(PayPal(), 10) == "PayPal 模拟付款：$10.00"
assert car.drive() == "汽车沿道路行驶"
print("学习检查全部通过：计算结果、实例状态、继承与多态均符合预期。")

学习检查全部通过：计算结果、实例状态、继承与多态均符合预期。


## 14. 对照教材与延伸阅读

本笔记沿用原文“过程式 → 函数 → 类”的主线，并演示新版新增的 Python 访问约定、支付多态与 `abc` 抽象类。原文中的伪代码拼写、返回语句和类名差异已统一修正；工资约束与三道练习是帮助理解概念的补充示例。

- 课程原文：[OOP approach Ver2.docx](source/OOP%20approach%20Ver2.docx)。
- Python 官方教程：[Classes](https://docs.python.org/3/tutorial/classes.html)，解释实例、继承、方法重写与名称改写。
- Python 官方文档：[abc — Abstract Base Classes](https://docs.python.org/3/library/abc.html)，说明抽象方法和实例化限制。

**自测：**不用看代码，解释为什么函数版本已经能解决幂求和；为什么 `__salary` 不是密码保险箱；为什么 `MissingPaymentProcessor()` 能创建，而 `IncompleteVehicle()` 不能创建。